In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import time

### Test Pytorch GPU

In [9]:
# --- Khởi tạo ---
print("Loading model...")
MODEL_NAME = "yolo11m.pt"
# Chỉ định sử dụng GPU CUDA. Đảm bảo PyTorch với CUDA đã được cài đặt đúng!
try:
    # device=0 tương đương với .to('cuda') nếu chỉ có 1 GPU
    model = YOLO(MODEL_NAME).to('cuda')
    print("Model loaded on GPU (CUDA).")
    # Có thể thêm dòng khởi động GPU nếu cần, nhưng thường không bắt buộc
    # _ = model(np.zeros((640, 640, 3), dtype=np.uint8), verbose=False)
except Exception as e:
    print(f"ERROR loading model on GPU: {e}")
    print("Ensure PyTorch with CUDA support is installed correctly and CUDA drivers are up to date.")
    print("Falling back to CPU.")
    model = YOLO(MODEL_NAME) # Tải trên CPU nếu có lỗi

Loading model...
Model loaded on GPU (CUDA).


### Final version

In [ ]:
def count_suitcases(video_path, model_name, counting_line):
    # --- Cấu hình mặc định ---
    TARGET_CLASS_ID = 28
    CONFIDENCE_THRESHOLD = 0.5
    LINE_THICKNESS = 4
    FONT_SCALE = 1.5
    FONT_THICKNESS = 3
    FRAME_SKIP = 5
    FORGET_THRESHOLD_FRAMES = 150
    TOLERANCE = 1000
    LINE_P1 = tuple(counting_line[0])
    LINE_P2 = tuple(counting_line[1])

    def get_point_side(point, line_p1, line_p2):
        x, y = point
        x1, y1 = line_p1
        x2, y2 = line_p2
        cross_product = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        if abs(cross_product) < TOLERANCE:
            return 0
        return 1 if cross_product > 0 else -1

    # --- Khởi tạo ---
    try:
        # device=0 tương đương với .to('cuda') nếu chỉ có 1 GPU
        model = YOLO(MODEL_NAME).to('cuda')
        print("Model loaded on GPU (CUDA).")
        # Có thể thêm dòng khởi động GPU nếu cần, nhưng thường không bắt buộc
        # _ = model(np.zeros((640, 640, 3), dtype=np.uint8), verbose=False)
    except Exception as e:
        print(f"ERROR loading model on GPU: {e}")
        print("Ensure PyTorch with CUDA support is installed correctly and CUDA drivers are up to date.")
        print("Falling back to CPU.")
        model = YOLO(MODEL_NAME) # Tải trên CPU nếu có lỗi

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Lỗi: Không thể mở video {video_path}")
        return

    counted_ids = set()
    last_known_side = {}
    last_seen_frame = {}
    count = 0
    frame_count = 0
    start_time = time.time()
    processed_frame_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        if frame_count % FRAME_SKIP != 0:
            continue

        processed_frame_count += 1
        current_frame_tracks = set()

        results = model.track(
            frame,
            persist=True,
            verbose=False,
            conf=CONFIDENCE_THRESHOLD,
            classes=[TARGET_CLASS_ID],
        )

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, track_ids):
                current_frame_tracks.add(track_id)
                last_seen_frame[track_id] = frame_count

                x1, y1, x2, y2 = box
                center_point = ((x1 + x2) // 2, (y1 + y2) // 2)

                current_side = get_point_side(center_point, LINE_P1, LINE_P2)
                prev_side = last_known_side.get(track_id)

                if current_side != 0:
                    if prev_side is not None and prev_side != current_side:
                        if track_id not in counted_ids:
                            count += 1
                            counted_ids.add(track_id)
                            print(f"Frame {frame_count}: Counted ID {track_id}. Total: {count}")
                    last_known_side[track_id] = current_side

                # Vẽ khung và ID
                color = (0, 0, 255) if track_id in counted_ids else (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, LINE_THICKNESS // 2)
                cv2.putText(
                    frame,
                    f"ID:{track_id} S:{current_side}",
                    (x1, y1 - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE * 0.8,
                    color,
                    FONT_THICKNESS,
                )
                cv2.circle(frame, center_point, 5, (0, 0, 255), -1)

        # --- Thuật toán quên đi ID cũ sau 150 frame ---
        ids_to_check = list(counted_ids)
        ids_to_forget = set()
        for c_id in ids_to_check:
            if c_id not in current_frame_tracks:
                last_seen = last_seen_frame.get(c_id, -1)
                if last_seen != -1 and (frame_count - last_seen) > FORGET_THRESHOLD_FRAMES:
                    ids_to_forget.add(c_id)
                    print(f"Frame {frame_count}: Forgetting ID {c_id}")

        counted_ids -= ids_to_forget

        for d_id in list(last_known_side.keys()):
            if d_id not in current_frame_tracks:
                last_known_side.pop(d_id, None)

        for f_id in ids_to_forget:
            last_seen_frame.pop(f_id, None)

        # --- Hiển thị thông tin ---
        cv2.line(frame, LINE_P1, LINE_P2, (255, 0, 0), LINE_THICKNESS)
        cv2.putText(
            frame,
            f"Count: {count}",
            (30, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            FONT_SCALE * 1.2,
            (0, 0, 255),
            FONT_THICKNESS + 1,
        )

        elapsed_time = time.time() - start_time
        if elapsed_time > 0:
            fps = processed_frame_count / elapsed_time
            cv2.putText(
                frame,
                f"FPS: {fps:.2f}",
                (frame.shape[1] - 250, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                FONT_SCALE,
                (0, 255, 0),
                FONT_THICKNESS,
            )

        scaled_frame = cv2.resize(frame, None, fx=0.4, fy=0.4)
        cv2.imshow("Suitcase Counter", scaled_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Final count: {count}")

In [10]:
count_suitcases(
    video_path="video1.mp4",
    model_name="yolo11m.pt",
    counting_line=np.array([[560, 441], [313, 894]], dtype=np.int32)
)

Model loaded on GPU (CUDA).
Frame 80: Counted ID 1. Total: 1
Frame 225: Counted ID 2. Total: 2
Frame 270: Counted ID 6. Total: 3
Frame 295: Counted ID 7. Total: 4
Frame 305: Forgetting ID 1
Frame 440: Forgetting ID 2
Frame 475: Counted ID 9. Total: 5
Frame 475: Forgetting ID 6
Frame 520: Forgetting ID 7
Frame 600: Counted ID 8. Total: 6
Final count: 6


In [11]:
count_suitcases(
    video_path="video4.mp4",
    model_name="yolo11m.pt",
    counting_line=np.array([[392, 400], [1475, 397]], dtype=np.int32)
)

Model loaded on GPU (CUDA).
Frame 90: Counted ID 1. Total: 1
Frame 250: Forgetting ID 1
Frame 305: Counted ID 4. Total: 2
Frame 340: Counted ID 6. Total: 3
Frame 475: Forgetting ID 4
Frame 480: Counted ID 3. Total: 4
Frame 495: Forgetting ID 6
Final count: 4
